# Module 5: Grounding with RAG and Reranking

⏱ **Estimated time:** 45 minutes

In this notebook, you'll give Nova Micro the relevant legislation when it answers, then score it on the same 50 questions, rubric and judge as Module 2.

## What You'll Do

1. Load 12 Acts into a Bedrock Knowledge Base backed by Amazon S3 Vectors
2. Look at what retrieval returns for a question
3. Score RAG with the top 10 passages
4. See a passage that retrieving more results can't reach, but reranking can
5. Score RAG with reranking
6. Compare every configuration measured so far

---
Copyright Amazon.com, Inc. or its affiliates. All Rights Reserved.  
SPDX-License-Identifier: MIT-0

## Step 0: Install the Notebook Dependencies

Run this first to install the required packages for the notebook.

In [ ]:
# Install the notebook's dependencies (requirements-notebooks.txt sits beside the notebook)
get_ipython().run_line_magic("pip", "install -q -r requirements-notebooks.txt")
print("✅ Dependencies installed.")

## Configuration

Sets up the variables and AWS resources used in this notebook.

In [ ]:
# Settings
import boto3
import json
import os
import time
import logging
from datetime import datetime, timezone

# Let Inspect AI run inside Jupyter
import nest_asyncio
nest_asyncio.apply()
logging.getLogger("asyncio").setLevel(logging.CRITICAL)

# Settings: the AWS region, and which dataset to use
REGION = os.environ.get("AWS_REGION", "us-east-1")
DOMAIN = "legal"
JURISDICTION = "au"

# How many questions to score, and how many passages to fetch for each one
NUM_EVAL_RECORDS = 50
TOP_K = 10                # passages that go into the prompt
RETRIEVE_WIDE = 50        # candidates reranking chooses from
MAX_CONNECTIONS = 8       # parallel requests to Bedrock
FORCE_REINGEST = False    # True to re-ingest even if the knowledge base has the corpus

# Where the questions and the legislation (the corpus) live in S3, and which models to use
EVAL_KEY = f"{DOMAIN}/eval-{DOMAIN}-{JURISDICTION}.jsonl"
CORPUS_PREFIX = f"{DOMAIN}/corpus/"
BASELINE_MODEL = "us.amazon.nova-micro-v1:0"
RERANK_MODEL_ID = "cohere.rerank-v3-5:0"
# Scores from every module are saved under this prefix so they can be compared later
CELL_RESULTS_PREFIX = "results/cells/"
NOTEBOOK_NAME = "Module-5-RAG"

print(f"Retrieval: {RETRIEVE_WIDE} candidates → rerank → top {TOP_K}")

Reads bucket names and the knowledge base ID from the stack outputs.

> This should print a `KnowledgeBaseId`. If it prints `None`, the retrieval steps
> below won't run.

In [ ]:
# AWS clients, and bucket names and the knowledge base IDs from the stack outputs
def stack_outputs():
    """Every CloudFormation output in the region, by key."""
    # The CloudFormation stack records the names of the resources it created
    cf = boto3.client("cloudformation", region_name=REGION)
    return {o["OutputKey"]: o["OutputValue"]
            for page in cf.get_paginator("describe_stacks").paginate()
            for s in page["Stacks"] for o in s.get("Outputs", [])}

# Look up the buckets, the knowledge base, and its data source (the S3 folder it reads from)
outputs = stack_outputs()
DATASET_BUCKET = outputs["DatasetBucketName"]
RESULTS_BUCKET = outputs["ResultsBucketName"]
KNOWLEDGE_BASE_ID = outputs.get("KnowledgeBaseId")
DATA_SOURCE_ID = outputs.get("DataSourceId")

# bedrock-agent manages the knowledge base; bedrock-agent-runtime searches it and reranks results
s3 = boto3.client("s3", region_name=REGION)
bedrock = boto3.client("bedrock", region_name=REGION)
bedrock_agent = boto3.client("bedrock-agent", region_name=REGION)
rt = boto3.client("bedrock-agent-runtime", region_name=REGION)

print(f"KnowledgeBaseId: {KNOWLEDGE_BASE_ID}")
print(f"Corpus:          s3://{DATASET_BUCKET}/{CORPUS_PREFIX}")

## Step 0a: The Model Being Tested

Untuned Nova Micro, the model Module 2 measured and Module 4 fine-tuned. Keeping
the model fixed means any change in score comes from retrieval alone.

In [ ]:
# The model under test: untuned Nova Micro
# The label is stored with each saved score so base and fine-tuned results can be told apart
MODEL_LABEL = "base"
# Inspect AI names models as provider/model-id
MODEL_UNDER_TEST = f"bedrock/{BASELINE_MODEL}"
print(f"Model under test: {MODEL_UNDER_TEST}")

## Step 0b: Load Earlier Results from S3

Every module saves its scores to `results/cells/` in S3, so the final table can
combine results across notebooks.

In [ ]:
# Save and load scores in results/cells/
def save_cell_result(cell, model_label, model_id, score, per_item, config=None):
    """Save one configuration's score to results/cells/."""
    # Record the score with enough detail to know how it was produced
    rec = {"cell": cell, "model_label": model_label, "model_id": model_id, "score": score,
           "n": len(per_item), "judge": "us.anthropic.claude-sonnet-4-6",
           "eval_key": EVAL_KEY, "notebook": NOTEBOOK_NAME,
           "timestamp": datetime.now(timezone.utc).isoformat(),
           "config": config or {}, "per_item": per_item}
    # A timestamp in the file name means each run is kept rather than overwritten
    key = f"{CELL_RESULTS_PREFIX}{cell}-{model_label}-{datetime.now(timezone.utc):%Y%m%d-%H%M%S}.json"
    s3.put_object(Bucket=RESULTS_BUCKET, Key=key, Body=json.dumps(rec).encode(),
                  ContentType="application/json")
    print(f"  ✓ saved s3://{RESULTS_BUCKET}/{key}")

def load_cell_results():
    """Newest full-size (50-item) result for each configuration."""
    newest = {}
    # Read every saved result file in the results bucket
    pages = s3.get_paginator("list_objects_v2").paginate(Bucket=RESULTS_BUCKET, Prefix=CELL_RESULTS_PREFIX)
    for obj in (o for page in pages for o in page.get("Contents", []) if o["Key"].endswith(".json")):
        rec = json.loads(s3.get_object(Bucket=RESULTS_BUCKET, Key=obj["Key"])["Body"].read())
        # Keep the newest run for each configuration and model, ignoring smaller test runs
        key = (rec.get("cell"), rec.get("model_label"))
        if rec.get("n", 0) >= NUM_EVAL_RECORDS and rec["timestamp"] > newest.get(key, {}).get("timestamp", ""):
            newest[key] = rec
    return newest

print("✅ Ready to proceed!")

**Load the scores saved by earlier modules.** The next cell lists the scores already in S3, such as the Module 2 baseline and Module 4's fine-tuned score.

> Any results that aren't found yet show as `—` in the final table.

In [ ]:
# Print the results already saved by earlier modules
prior = load_cell_results()
# One line per configuration: model label, configuration name, score and when it ran
for (cell, label), rec in sorted(prior.items()):
    print(f"  {label:<10} {cell:<16} {rec['score']:.3f}  ({rec['timestamp'][:16]})")
if not prior:
    print("No earlier results yet. Module 2 saves the no-retrieval baseline.")

## Step 1: Ingest the Corpus into the Knowledge Base

Starts an ingestion job over the 12 Acts in the dataset bucket. Bedrock splits the text into
passages, turns each into a vector with Titan Text Embeddings V2, and stores them
in S3 Vectors.

> **Skipped if the knowledge base already holds the corpus**, which saves 5–10
minutes. Set `FORCE_REINGEST = True` to run it anyway.

In [ ]:
# Ingest the Acts from S3 into the knowledge base (skipped if already done)
# An ingestion job reads the documents from S3, splits them into passages (chunks),
# turns each chunk into an embedding (a list of numbers that captures its meaning)
# and stores the embeddings in the S3 Vectors index so they can be searched
# Check for an earlier job that completed and actually read documents
jobs = bedrock_agent.list_ingestion_jobs(knowledgeBaseId=KNOWLEDGE_BASE_ID,
                                         dataSourceId=DATA_SOURCE_ID)["ingestionJobSummaries"]
already_done = any(j["status"] == "COMPLETE" and j.get("statistics", {}).get("numberOfDocumentsScanned")
                   for j in jobs)
# Count the Act files in the corpus folder, to catch a missing data load
n_acts = s3.list_objects_v2(Bucket=DATASET_BUCKET, Prefix=CORPUS_PREFIX).get("KeyCount", 0)

if n_acts == 0:
    print(f"✗ No Acts in s3://{DATASET_BUCKET}/{CORPUS_PREFIX}. Load the data first (Prerequisite Setup).")
elif already_done and not FORCE_REINGEST:
    print("✓ The knowledge base already has the corpus. Skipping ingestion.")
else:
    # Start a new job, then check its status every 30 seconds until it finishes
    job_id = bedrock_agent.start_ingestion_job(knowledgeBaseId=KNOWLEDGE_BASE_ID,
                                               dataSourceId=DATA_SOURCE_ID)["ingestionJob"]["ingestionJobId"]
    print("Ingesting (5-10 minutes)...")
    while True:
        job = bedrock_agent.get_ingestion_job(knowledgeBaseId=KNOWLEDGE_BASE_ID,
                                              dataSourceId=DATA_SOURCE_ID, ingestionJobId=job_id)["ingestionJob"]
        if job["status"] in ("COMPLETE", "FAILED"):
            break
        time.sleep(30)
    print(f"✓ Ingestion complete: {job.get('statistics')}" if job["status"] == "COMPLETE"
          else f"✗ Ingestion failed: {job.get('failureReasons')}")

## Step 2: Inspect What Retrieval Returns

The cell below sends one question to the knowledge base, which searches by meaning
(semantic search) and returns the 10 passages closest to the question. It prints
each passage with the Act it came from. This is the same search plain RAG uses.

> Check whether all 10 passages are about what the question asks, or whether some
> are similar-sounding passages from elsewhere. 

In [ ]:
# Retrieve the top passages for one question and print them
DEMO_QUESTION = ("Under section 180(1) of the Corporations Act 2001 (Cth), what "
                 "standard must a director exercise in their duties?")

def retrieve(query, n):
    """The n passages most similar to the query."""
    # The knowledge base embeds the question and returns the chunks whose embeddings are closest
    return rt.retrieve(knowledgeBaseId=KNOWLEDGE_BASE_ID, retrievalQuery={"text": query},
                       retrievalConfiguration={"vectorSearchConfiguration": {"numberOfResults": n}}
                       ).get("retrievalResults", [])

print(f"Question: {DEMO_QUESTION}\n")
# For each passage, show its similarity score, the Act it came from (taken from the
# file name) and the first 100 characters with line breaks collapsed
for i, r in enumerate(retrieve(DEMO_QUESTION, TOP_K), 1):
    act = r["location"]["s3Location"]["uri"].rsplit("/", 1)[-1].removesuffix(".txt")
    print(f"  {i:>2}. [{r['score']:.3f}] {act}: {' '.join(r['content']['text'].split())[:100]}...")

## Step 3: Configure the Rubric and Judge

The cell below sets up the same rubric and judge model as Module 2, so this module's
scores can be compared directly with the baseline.

In [ ]:
# The rubric and judge
from inspect_ai import Task, task, eval as inspect_eval
from inspect_ai.dataset import FieldSpec, json_dataset
from inspect_ai.model import get_model, GenerateConfig
from inspect_ai.solver import generate, solver, Generate, TaskState
from inspect_ai.scorer import model_graded_qa

# The judge's rubric, identical in Modules 2, 4, 5 and 6
LEGAL_INSTRUCTIONS = """Grade the submitted answer against the reference answer for a question about Australian law.

Mark the answer CORRECT only if ALL of the following hold:
- It reaches the correct legal conclusion (e.g. liable/not liable, valid/void).
- It cites the correct legislation or section where the reference answer does.
- It is specific to Australian law, not generic or another jurisdiction's law.

Mark it INCORRECT if the legal conclusion is wrong, the cited provision is wrong
or fabricated, or the answer is generic rather than Australian-specific.

First explain your reasoning, then end with your grade in exactly this format:
"GRADE: C" if correct or "GRADE: I" if incorrect."""

# The judge model is a larger model that grades each answer against the rubric
# Temperature 0 makes its grading as consistent as possible between runs
grader = get_model("bedrock/us.anthropic.claude-sonnet-4-6",
                   config=GenerateConfig(temperature=0.0))

print("✅ Ready to proceed!")

**Define the scoring function.** The cell below defines two functions that the
later steps use:

- `score_cell` scores one configuration, such as plain RAG. It asks the model
  each of the 50 questions, has the judge grade every answer against the rubric,
  turns the grades into a score (the share of answers marked correct), and saves
  the result to S3.
- `print_comparison` reads the saved scores and prints them side by side in a table.

In [ ]:
# Score one configuration on the 50 questions and save the result
def score_cell(cell, task_solver=None, *, config=None, save=True):
    """Score one configuration on the 50 questions. Returns (score, per_item)."""
    # An Inspect task pairs the questions with a solver (the steps that produce an answer)
    # and a scorer (here, the judge model and rubric)
    @task
    def cell_task():
        return Task(
            dataset=json_dataset(f"s3://{DATASET_BUCKET}/{EVAL_KEY}",
                                 FieldSpec(input="prompt", target="referenceResponse", metadata=["category"]),
                                 limit=NUM_EVAL_RECORDS),
            solver=task_solver or generate(),
            scorer=model_graded_qa(instructions=LEGAL_INSTRUCTIONS, model=grader),
        )

    # Run the evaluation against the model under test, with logs written per configuration
    log = inspect_eval(cell_task(), model=MODEL_UNDER_TEST, log_dir=f"./inspect-logs/{cell}-{MODEL_LABEL}",
                       max_tokens=1024, max_connections=MAX_CONNECTIONS, temperature=0.0)[0]
    if log.status != "success" or not log.samples:
        print(f"  ✗ {cell}: the evaluation failed. Check the log above.")
        return None, []

    # Turn each judge grade into 1 (C, correct) or 0 (anything else)
    per_item = [{"id": s.id if s.id is not None else i,
                  "category": (s.metadata or {}).get("category", "unknown"),
                  "score": 1.0 if s.scores and str(next(iter(s.scores.values())).value).upper().startswith("C") else 0.0}
                for i, s in enumerate(log.samples, 1)]
    # The overall score is the fraction of questions marked correct
    score = sum(p["score"] for p in per_item) / len(per_item)
    print(f"  {cell} ({MODEL_LABEL}): {score:.3f}  (n={len(per_item)})")
    if save:
        save_cell_result(cell, MODEL_LABEL, MODEL_UNDER_TEST, score, per_item, config)
    return score, per_item

def print_comparison(cell_order):
    """Print the newest score for each configuration, base and fine-tuned."""
    latest = load_cell_results()
    print(f"  {'Configuration':<36} {'Base':>8} {'Fine-tuned':>12}")
    print("  " + "-" * 58)
    # One row per configuration, with the base and fine-tuned scores side by side
    for cell, label in cell_order:
        b, f = (latest.get((cell, m), {}).get("score") for m in ("base", "finetuned"))
        print(f"  {label:<36} {f'{b:.3f}' if b is not None else '—':>8} {f'{f:.3f}' if f is not None else '—':>12}")
    print("\n  — = not measured yet. Repeat runs vary by about 0.04.")
    return latest

print("✅ Ready to proceed!")

## Step 4: The RAG Solver

In Inspect, the *solver* decides how a question is answered, so adding retrieval
only changes the solver.

- `rerank=False`: retrieve the `TOP_K` most similar passages.
- `rerank=True`: retrieve `RETRIEVE_WIDE` candidates, re-sort them with Cohere
  Rerank 3.5, and keep the best `TOP_K`.

In [ ]:
# Rerank candidate passages with Cohere Rerank 3.5
# A reranker is a model that reads the question and each passage together and scores how
# well the passage answers it. This is slower than similarity search but more accurate.
RERANK_MODEL_ARN = f"arn:aws:bedrock:{REGION}::foundation-model/{RERANK_MODEL_ID}"

def rerank_candidates(rt, query, results, top_k):
    """Re-sort the candidates by relevance and keep the best top_k. Returns (kept, moves)."""
    # Send the question and the candidate passages (each cut to 4,000 characters) to the reranker
    out = rt.rerank(
        queries=[{"type": "TEXT", "textQuery": {"text": query}}],
        sources=[{"type": "INLINE", "inlineDocumentSource": {
                      "type": "TEXT", "textDocument": {"text": r["content"]["text"][:4000]}}}
                 for r in results],
        rerankingConfiguration={"type": "BEDROCK_RERANKING_MODEL", "bedrockRerankingConfiguration": {
            "numberOfResults": min(top_k, len(results)),
            "modelConfiguration": {"modelArn": RERANK_MODEL_ARN}}},
    )
    # The reranker returns positions in the original list, best first
    kept = [results[item["index"]] for item in out["results"]]
    # Record each passage's old (similarity) rank and new rank, to show what reranking changed
    moves = [{"new_rank": n, "vector_rank": item["index"] + 1,
              "relevance": round(item.get("relevanceScore", 0.0), 4)}
             for n, item in enumerate(out["results"], 1)]
    return kept, moves

print("✅ Ready to proceed!")

**The solver.** Retrieves passages, optionally reranks them, adds them to the
prompt, then asks the model. It also records what was retrieved, which Step 6 uses.

In [ ]:
# The RAG solver: retrieve, optionally rerank, add the passages to the prompt, then answer
def make_rag_solver(retrieved_log, *, top_k=TOP_K, rerank=False):
    """An Inspect solver that answers from retrieved legislation."""
    # A solver is the code Inspect runs for each question to produce the model's answer
    @solver
    def rag_solver():
        async def solve(state: TaskState, generate: Generate) -> TaskState:
            # With reranking, fetch a wider pool of candidates for the reranker to choose from
            query = state.user_prompt.text
            results = retrieve(query, RETRIEVE_WIDE if rerank else top_k)
            moves = None
            if rerank and results:
                results, moves = rerank_candidates(rt, query, results, top_k)
            retrieved_log.append({"question": query[:160], "moves": moves})

            # Put the passages in the prompt ahead of the question, then call the model
            context = "\n\n---\n\n".join(r["content"]["text"] for r in results)
            state.user_prompt.text = (
                "Use the following Australian legislation extracts to answer the "
                "question. If they are relevant, cite the specific section.\n\n"
                f"<legislation>\n{context}\n</legislation>\n\n"
                f"Question: {query}"
            )
            return await generate(state)
        return solve
    return rag_solver()

print("✅ Ready to proceed!")

## Step 5: Score Plain RAG

Retrieves the `TOP_K` most similar passages and adds them to the prompt, with no
reranking.

> This takes about 2 minutes to run. Compare the score with the baseline from Module 2.

In [ ]:
# Score plain RAG
# rag_log collects what was retrieved for each question
rag_log = []
rag_score, _ = score_cell("rag", make_rag_solver(rag_log), config={"top_k": TOP_K, "rerank": False})

## Step 6: Why Retrieving More Isn't the Answer

The Corporations Act imposes a near-identical "care and diligence" duty in about
ten places, on different kinds of officeholder. Searching by meaning can't tell
these apart from s180(1), the one the demo question asks about.

> Compare where the real s180(1) passage ranks before and after reranking.

**First, how to spot the right passage.** Passage boundaries don't line up with
section boundaries: the passage holding s180(1) starts partway through the
previous section. So the code matches on s180(1)'s actual wording, and
`excerpt_around` shows the relevant part of each passage.

In [ ]:
# The opening words of s180(1), used to spot its passage
S180_TEXT = ("a director or other officer of a corporation must exercise their powers "
             "and discharge their duties with the degree of care and diligence")

def excerpt_around(text, needle, width=90):
    """The part of a passage around the needle."""
    # Find the phrase ignoring case, falling back to the start of the passage if it isn't there
    pos = max(text.lower().find(needle.lower()), 0)
    return "..." + text[max(0, pos - 20):pos + width] + "..."

print("✅ Ready to proceed!")

**Now the demonstration.** Retrieve 50 candidates, find where s180(1) ranks by
similarity, then rerank the *same* candidates and look again.

> Note where s180(1) ranks by similarity, whether it falls inside the top `TOP_K`
> that plain RAG uses, and where it moves after reranking.

In [ ]:
# Show where s180(1) ranks by similarity, then after reranking
# Fetch the wide pool of candidates and collapse whitespace so the text can be matched
candidates = retrieve(DEMO_QUESTION, RETRIEVE_WIDE)
texts = [" ".join(r["content"]["text"].split()) for r in candidates]
# Find the rank of the passage containing s180(1), if it was retrieved at all
s180_rank = next((i for i, t in enumerate(texts, 1) if S180_TEXT in t.lower()), None)

# List every passage that mentions "care and diligence", since many Acts impose a similar duty
print("Passages imposing a 'care and diligence' duty, by similarity rank:\n")
for i, t in enumerate(texts, 1):
    if "care and diligence" in t.lower():
        mark = "  ← s180(1)" if i == s180_rank else ""
        print(f"  rank {i:>2}: {excerpt_around(t, S180_TEXT if mark else 'care and diligence')}{mark}")

# Rerank the same candidates and see where s180(1) lands
kept, moves = rerank_candidates(rt, DEMO_QUESTION, candidates, TOP_K)
new_rank = next((m["new_rank"] for m in moves if m["vector_rank"] == s180_rank), None)
print(f"\ns180(1): similarity rank {s180_rank} → after reranking {new_rank or 'not in the top ' + str(TOP_K)}")

## Step 7: Score RAG with Reranking

Retrieves 50 candidates, reranks them, and adds the best `TOP_K` to the prompt.
Only the ordering changes from Step 5.

> ❗️ The scoring may take ~2 minutes to run.

In [ ]:
# Score RAG with reranking, and count passages reranking promoted
rerank_log = []
rerank_score, _ = score_cell("rag-rerank", make_rag_solver(rerank_log, rerank=True),
                             config={"top_k": TOP_K, "retrieve_wide": RETRIEVE_WIDE,
                                     "rerank": True, "rerank_model": RERANK_MODEL_ID})
# A passage was promoted if its similarity rank was below the top 10 but reranking kept it
promoted = sum(m["vector_rank"] > TOP_K for e in rerank_log for m in (e["moves"] or []))
print(f"\n{promoted} passage(s) came from outside the top {TOP_K} and were moved into the prompt by reranking.")

## Step 8 (Optional): The Cost and Time Variant

Because reranking picks better passages, you may be able to send fewer of them.
This step checks whether reranking to the top 5 can match plain RAG's top 10,
with half as many passages in the prompt.

> This cell is off by default. Set `RUN_TOP5_VARIANT = True` to run it. It takes
> about 2.5 minutes.

In [ ]:
# Optional: score reranking to the top 5
# Off by default to save time; tests whether fewer, better passages help
RUN_TOP5_VARIANT = False

if RUN_TOP5_VARIANT:
    score_cell("rag-rerank-top5", make_rag_solver([], top_k=5, rerank=True),
               config={"top_k": 5, "retrieve_wide": RETRIEVE_WIDE, "rerank": True,
                       "rerank_model": RERANK_MODEL_ID})
else:
    print("Skipped. Set RUN_TOP5_VARIANT = True to run it.")

## Step 9: Which Technique Closes the Gap?

Combines every full-size result in S3, including Module 2's baseline and Module 4's
fine-tuned score. A `—` means not measured yet.

> Compare each retrieval step with the one before it, and with Module 4's fine-tuned score.

In [ ]:
# Print every configuration's score side by side
# Each pair is the saved configuration name and the label shown in the table
latest = print_comparison([
    ("none",            "No retrieval"),
    ("rag",             f"+ RAG (top {TOP_K})"),
    ("rag-rerank",      f"+ RAG + reranking (top {TOP_K})"),
    ("rag-rerank-top5", "   + RAG + reranking (top 5)"),
])

## What to Observe

1. **Retrieval compared with fine-tuning.** Compare each retrieval step with the
   baseline and with Module 4's fine-tuned score.
2. **Ranking.** In Step 6, check whether s180(1) made it into the top 10 by
   similarity, and how reranking changed its position.
3. **Whether the model uses what it's given.** Look for answers that are wrong even
   though the right passage was in the prompt. The s1043A penalty question is a good
   one to check: if the model still gets it wrong, that's a limit of the model, not
   of retrieval.
4. **Check the statistics.** Compare the gain from baseline to RAG with the gain
   from reranking alone. At 50 questions, a small gain may not stand out from
   run-to-run variation.

**Next:** Module 6 tries a different kind of search on questions like s1043A.

## Success Criteria

Before moving on, verify that you have:

- [ ] 12 Acts ingested into the Knowledge Base (or a skip message confirming they were)
- [ ] Score recorded for plain RAG on the full 50 items
- [ ] Score recorded for RAG + reranking
- [ ] Compared the s180(1) rank before and after reranking in Step 6
- [ ] A comparison table including Module 2's baseline row

## Key Takeaways

- A factual gap is a job for retrieval rather than fine-tuning.
- Retrieving more passages doesn't help when the right one is ranked too low;
  reranking does.
- The right passage in the prompt doesn't guarantee the model uses it.
- Reranking can improve accuracy *and* shrink the prompt.
- You can't retrieve what isn't in the corpus.

## Next Steps

Open **Module-6-Agentic-Search.ipynb**, which tries a different kind of search on
questions like the s1043A penalty question.